In [ ]:
""" 
Script to generate the source data for Supplementary Figure 04.
"""
import pandas as pd
import os

datadir = os.path.abspath(os.path.join(os.path.dirname( os.getcwd() ), '.', 'data'))
print(f'Data directory: {datadir}')

maindir = '' # define the main directory where the BIDS datasets are stored

pipver = ''
task = 'rest'
phases = ['p2', 'p5']

lfreq = 0.1 #Hz
hfreq = 145.0 #Hz
fsample = 300.0 #Hz
frange = f"{round(lfreq, 1)}-{int(hfreq)}Hz"

trans = True # Whether to use head transformation or not
zmm = 44 # destination z coordinate head position in mm

icselection = 'ecg04eog08' # 'allbutecg04' #'eog08' #
proc = 'filt' + icselection #'sss' #'clean'

# Directories and file names
deriv_folder = f'aperiodic_filt{frange}_fs{int(fsample)}Hz_trans_z{zmm}mm'
taskref = 'rest'
phaseref = 'p5'
armref = 1
bids_project_folder = f'BIDS_long_{phaseref}_{taskref}_arm{armref}'
deriv_root = os.path.join(maindir, bids_project_folder,
                          'derivatives', deriv_folder)
statsdir = os.path.join(deriv_root, 'stats')

loaddir = statsdir

fitting_param = 'finley' 

parameters = ['r_squared', 'theta_peak_freq', 'theta_band_power'] 
megtype = 'grad'

for p, parameter in enumerate(parameters):

    df_plot = pd.DataFrame(columns=[parameter, 'Aperiodic_mode', 'Counts'])

    for hasknee in [True, False]:
        withknee = 'knee' if hasknee else ''
        
        varfile = os.path.join(loaddir, f'aperiodic_stier_{proc}_{fitting_param}{withknee}_{megtype}{parameter}_2betas.tsv')

        df_vars = pd.read_csv(varfile, sep='\t').set_index(['row'])

        channels = [c for c in df_vars.columns if c.startswith('MEG') and (c.endswith('1') or c.endswith('2') or c.endswith('3'))]

        if df_plot.empty:
            df_plot = pd.DataFrame({parameter: df_vars[channels].mean(axis=1), 'Aperiodic_mode': ['Knee' if hasknee else 'Fixed']*df_vars.shape[0], 'Counts': df_vars[channels].notna().sum(axis=1)})
        
        else:
            df_plot = pd.concat([df_plot, pd.DataFrame({parameter: df_vars[channels].mean(axis=1), 'Aperiodic_mode': ['Knee' if hasknee else 'Fixed']*df_vars.shape[0], 'Counts': df_vars[channels].notna().sum(axis=1)})], ignore_index=True)
        
    df_plot.to_csv(os.path.join(datadir, f'supp_figure04_{parameter}_source_data.csv'), index=False)